In [ ]:
!pip uninstall -y unsloth unsloth-zoo
!pip cache purge
!pip install -U --no-cache-dir "unsloth[colab-new]" unsloth-zoo

Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 172.2 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [ ]:
!pip install --no-cache-dir torch torchvision torchaudio

In [ ]:
!pip install huggingface_hub

In [ ]:
import pandas as pd
import json
from huggingface_hub import HfApi, login
from google.colab import files
import io

In [ ]:
print("upload csv file")
uploaded = files.upload()

upload csv file


Saving callme.csv to callme.csv


In [ ]:
print('Type of input CSV :', type(uploaded))
print('Data :', uploaded)
print('Data Len :', len(uploaded))
print('Print by item')
for n in iter(uploaded) :   # 딕셔너리에서 iter() 키(key)만 순회합니다
  print(n)                  # 첫번째 key값은 file 명

Type of input CSV : <class 'dict'>
Data : {'callme.csv': b'instruction,input,response\r\nanswer to input,What is the CallCrazy service?,The CallCrazy service is a delivery service recently launched in Korea.\r\nanswer to input,What can I order with CallCrazy?,"We can deliver fast food, Japanese food, Chinese food, snacks, food ingredients, general items, etc."\r\nanswer to input,How much does the CallCrazy service cost?,"The basic fee is 5,000 won, and 1,000 won is added per km"\r\nanswer to input,What are the features of CallCrazy service?,We show the current location of the shipment on a map and deliver the product inexpensively and reliably.\r\nanswer to input,Does CallCrazy have a mileage system?,"Yes, CallCrazy mileage accumulates by 5% each time you use it, and you can use it when your mileage exceeds 10,000 won"\r\nanswer to input,How can I use CallCrazy service?,"\nInstall the CallCrazy app or call 777-7777."\r\nanswer to input,Can CallCrazy be used overseas?,CallCrazy service 

io.BytesIO는 Python의 입출력(IO) 모듈로, 바이너리 데이터(bytes)를 메모리에서 파일처럼 처리할 수 있게 합니다. uploaded[filename]은 딕션어리의 value 값으로 csv 형식으로 callme data값이 들어가 있다.

In [ ]:
file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='cp949')
print(df.head(5))
print("\n컬럼:", df.columns)
print("\n행 수:", len(df))

       instruction                                        input  \
0  answer to input               What is the CallCrazy service?   
1  answer to input             What can I order with CallCrazy?   
2  answer to input    How much does the CallCrazy service cost?   
3  answer to input  What are the features of CallCrazy service?   
4  answer to input        Does CallCrazy have a mileage system?   

                                            response  
0  The CallCrazy service is a delivery service re...  
1  We can deliver fast food, Japanese food, Chine...  
2  The basic fee is 5,000 won, and 1,000 won is a...  
3  We show the current location of the shipment o...  
4  Yes, CallCrazy mileage accumulates by 5% each ...  

컬럼: Index(['instruction', 'input', 'response'], dtype='object')

행 수: 11


In [ ]:
# CSV를 JSONL로 변환

# 1. CSV 파일 읽기
file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='cp949')

# 2. 모든 열의 데이터를 문자열로 변환
df = df.astype(str)

# 3. DataFrame을 JSON Lines (JSONL) 형식으로 변환
jsonl_file_name = 'call_me.jsonl'
jsonl_path = f'/content/{jsonl_file_name}'

with open(jsonl_path, 'w', encoding='utf-8') as f:
    for _, row in df.iterrows():
        json.dump(row.to_dict(), f, ensure_ascii=False)
        print(row.to_dict())
        f.write('\n')

print(f"데이터셋이 성공적으로 정리되어 '{jsonl_file_name}' 파일로 저장되었습니다.")

# 4. 저장된 JSONL 파일의 처음 몇 줄 확인
print("\n저장된 JSONL 파일의 처음 5줄:")
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

# 5. 데이터셋 정보 출력
print(f"\n총 행 수: {len(df)}")
print(f"컬럼: {', '.join(df.columns)}")

{'instruction': 'answer to input', 'input': 'What is the CallCrazy service?', 'response': 'The CallCrazy service is a delivery service recently launched in Korea.'}
{'instruction': 'answer to input', 'input': 'What can I order with CallCrazy?', 'response': 'We can deliver fast food, Japanese food, Chinese food, snacks, food ingredients, general items, etc.'}
{'instruction': 'answer to input', 'input': 'How much does the CallCrazy service cost?', 'response': 'The basic fee is 5,000 won, and 1,000 won is added per km'}
{'instruction': 'answer to input', 'input': 'What are the features of CallCrazy service?', 'response': 'We show the current location of the shipment on a map and deliver the product inexpensively and reliably.'}
{'instruction': 'answer to input', 'input': 'Does CallCrazy have a mileage system?', 'response': 'Yes, CallCrazy mileage accumulates by 5% each time you use it, and you can use it when your mileage exceeds 10,000 won'}
{'instruction': 'answer to input', 'input': 'H

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('Huggingface')

HuggingFace에 로그인

In [ ]:
login(token=HF_TOKEN)

In [ ]:
# JSONL 파일을 HuggingFace에 업로드
api = HfApi()
api.upload_file(
    path_or_fileobj=jsonl_path,
    path_in_repo=jsonl_file_name,  # Hugging Face 저장소에 업로드될 파일 이름
    repo_id="JaeminKim/callme",
    repo_type="dataset"
)

print(f"파일 '{jsonl_file_name}'이 성공적으로 업로드되었습니다!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


파일 'call_me.jsonl'이 성공적으로 업로드되었습니다!


 %%capture
Jupyter/Colab에서 사용하는 매직 명령어입니다.
이 명령어는 셀의 출력 결과를 숨기는 역할을 합니다.

Unsloth는 사용자의 딥러닝 워크플로를 간소화하고 가속화하기 위한 Python 라이브러리입니다. 특히 Colab과 같은 환경에서 딥러닝 실험 및 애플리케이션을 구축하기 쉽게 설계되었습니다

In [ ]:
model_name = 'unsloth/Llama-3.2-3B-Instruct'
max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = None           # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True    # Use 4bit quantization to reduce memory usage. Can be False.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
from transformers import pipeline

# pipeline에 로드된 model과 tokenizer 전달
text_generator = pipeline(
    "text-generation",    # 다른 pipeline도 많음, keyword
    model=model,  # 직접 로드한 모델 전달
    tokenizer=tokenizer,
    max_new_tokens=128,
    #device=0
)
def get_response(prompt) :
  sequences = text_generator(prompt)
  gen_text = sequences[0]['generated_text']

  return gen_text

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
prompt = 'What is Machine Learning?'
response = get_response(prompt)
response

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention 

'What is Machine Learning? and its Applications in Healthcare\n**What is Machine Learning?**\n\nMachine learning is a subset of artificial intelligence that involves training algorithms to learn patterns and relationships within data, without being explicitly programmed. In other words, machine learning allows computers to automatically improve their performance on a task without being explicitly told how to perform the task.\n\nThe process of machine learning involves several steps:\n\n1.  **Data Collection**: Gathering relevant data that is relevant to the task at hand.\n2.  **Data Preprocessing**: Cleaning and preparing the data for use in machine learning algorithms.\n3.  **Model Training**: Using the preprocessed data to train a machine'

In [ ]:
prompt = '서울의 유명 관광지는?'
response = get_response(prompt)
response

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"서울의 유명 관광지는? Seoul's famous tourist attractions?\n Seoul의 유명 관광지는? Seoul's famous tourist attractions?\n*   **가산 (Gasan)**: Seoul의 가장 높은 산으로, 200m의 고도에 위치해 있습니다. 가산은 관광객들이 즐기기 위해 다양한 활동을 제공합니다.\n*   **한강 (Han River)**: 한강은 서울의 중심부에 위치해 있으며, 한강 공원과 한강을 자랑합니다. 관광객들이 한강에서 수영, 보트를 즐기는 곳으로도 사용됩니다.\n*   **인cheon (Incheon"

In [ ]:
prompt = 'What is CallCrazy service?'
response = get_response(prompt)
response

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"What is CallCrazy service? CallCrazy is a call-blocking service provided by RingCentral, a leading provider of cloud-based communication solutions. The service allows users to block unwanted or unwanted calls from being connected to their phones. Here's what you need to know about CallCrazy:\n**How it works:**\nCallCrazy uses advanced technology to identify and block unwanted calls. The service can block calls from specific phone numbers, numbers that have been previously blocked, or even entire phone number ranges.\n\n**Features:**\n\n1. **Customizable block lists:** You can create and manage your own block lists, allowing you to block specific phone numbers or numbers that have been"

We support Llama, Mistral, Phi-3, Gemma, Yi, DeepSeek, Qwen, TinyLlama, Vicuna, Open Hermes etc
We support 16bit LoRA or 4bit QLoRA. Both 2x faster.
max_seq_length can be set to anything, since we do automatic RoPE Scaling via kaiokendev's method.
[NEW] We make Gemma-2 9b / 27b 2x faster! See our Gemma-2 9b notebook
[NEW] To finetune and auto export to Ollama, try our Ollama notebook

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.4.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


[NOTE] Remember to add the EOS_TOKEN to the tokenized output!! Otherwise you'll get infinite generations!

In [ ]:
alpaca_prompt = """제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # EOS_TOKEN을 정의해야 합니다. tokenizer는 사전에 정의되어 있어야 합니다.

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    responses = examples["response"]
    texts = []

    for instruction, input, response in zip(instructions, inputs, responses):
        # EOS_TOKEN을 추가하여 생성이 무한히 진행되는 것을 방지합니다.
        text = alpaca_prompt.format(instruction, input, response) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

from datasets import load_dataset

dataset = load_dataset("JaeminKim/callme", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

# 첫 번째 예시를 출력하여 확인
print(dataset[0]['text'])

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

call_me.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/11 [00:00<?, ? examples/s]

Map:   0%|          | 0/11 [00:00<?, ? examples/s]

제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
answer to input

### Input:
What is the CallCrazy service?

### Response:
The CallCrazy service is a delivery service recently launched in Korea.<|eot_id|>


Train the model
Now let's use Huggingface TRL's SFTTrainer! More docs here: TRL SFT docs. We do 60 steps to speed things up, but you can set num_train_epochs=1 for a full run, and turn off max_steps=None. We also support TRL's DPOTrainer!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/11 [00:00<?, ? examples/s]

In [ ]:
#@title Show current memory stats
import torch

gpu_stats = torch.cuda.get_device_properties(0)
# 바이트 → 킬로바이트 → 메가바이트 → 기가,  소수점 3자리까지 표현
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
2.457 GB of memory reserved.


In [ ]:
import wandb
wandb.init(mode="disabled")
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11 | Num Epochs = 100 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.695094
2,3.695110
3,3.633601
4,3.404664
5,3.099999
6,2.746890
7,2.340835
8,1.969278
9,1.611525
10,1.315309


In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

133.333 seconds used for training.
2.22 minutes used for training.
Peak reserved memory = 3.543 GB.
Peak reserved memory for training = 1.086 GB.
Peak reserved memory % of max memory = 24.329 %.
Peak reserved memory for training % of max memory = 7.457 %.


Inference

Let's run the model! You can change the instruction and input - leave the output blank!

In [ ]:
# alpaca_prompt 정의
alpaca_prompt = """제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

# FastLanguageModel 설정 (이 부분은 그대로 유지)
FastLanguageModel.for_inference(model)

# 원본 dataset 로드
original_dataset = load_dataset("JaeminKim/callme", split="train")

# dataset에서 무작위로 하나의 예제 선택
import random
random_index = random.randint(0, len(original_dataset) - 1)
example = original_dataset[random_index]

# 입력 준비
instruction = example['instruction']
input_text = example['input']

# 토큰화
inputs = tokenizer(
    [
        alpaca_prompt.format(
            instruction=instruction,
            input=input_text
        )
    ],
    return_tensors="pt"
).to("cuda")

# 생성
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)

# 디코딩 및 출력
generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("생성된 응답:")
print(generated_text)

# 실제 응답 출력 (비교를 위해)
print("\nData Set 데이터:")
print(example['response'])

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


생성된 응답:
제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
answer to input

### Input:
What is the CallCrazy service?

### Response:
The CallCrazy service is a delivery service recently launched in Korea.

Data Set 데이터:
The CallCrazy service is a delivery service recently launched in Korea.


You can also use a TextStreamer for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
from datasets import load_dataset
from transformers import TextStreamer

# alpaca_prompt 정의
alpaca_prompt = """제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.
### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

# FastLanguageModel 설정
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# 원본 dataset 로드
original_dataset = load_dataset("JaeminKim/callme", split="train")

# dataset에서 무작위로 하나의 예제 선택
import random
random_index = random.randint(0, len(original_dataset) - 1)
example = original_dataset[random_index]

# 입력 준비 (선택된 예제에서 가져옴)
instruction = example['instruction']
input_text = example['input']

# 토큰화
inputs = tokenizer(
    [
        alpaca_prompt.format(
            instruction=instruction,
            input=input_text
        )
    ],
    return_tensors="pt"
).to("cuda")

# TextStreamer 설정
text_streamer = TextStreamer(tokenizer)

# 생성
outputs = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True
)

# 생성된 텍스트 출력
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n생성된 응답:")
print(generated_text)


# 실제 응답 출력 (비교를 위해)
print("\nData Set 데이터:")
print(example['response'])

<|begin_of_text|>제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.
### Instruction:
answer to input

### Input:
Can CallCrazy be used overseas?

### Response:


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CallCrazy service is currently only available in Korea.<|eot_id|>

생성된 응답:
제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.
### Instruction:
answer to input

### Input:
Can CallCrazy be used overseas?

### Response:
CallCrazy service is currently only available in Korea.

Data Set 데이터:
CallCrazy service is currently only available in Korea.


In [ ]:
# 입력 준비 (선택된 예제에서 가져옴)
instruction = 'answer to input'
input_text1 = 'What is the price of CallCrazy service?'
input_text2 = 'CallCrazy 서비스는 어떤 것들을 배달하나요?'

# 토큰화 #1
inputs = tokenizer(
    [
        alpaca_prompt.format(
            instruction=instruction,
            input=input_text1
        )
    ],
    return_tensors="pt"
).to("cuda")


# 생성
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True
)

# 생성된 텍스트 출력
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n생성된 응답:")
print(generated_text)


# 토큰화 #2
inputs = tokenizer(
    [
        alpaca_prompt.format(
            instruction=instruction,
            input=input_text2
        )
    ],
    return_tensors="pt"
).to("cuda")


# 생성
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True
)

# 생성된 텍스트 출력
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n생성된 응답:")
print(generated_text)


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



생성된 응답:
제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.
### Instruction:
answer to input

### Input:
What is the price of CallCrazy service?

### Response:
The basic fee is 5,000 won, and 1,000 won is added per km

생성된 응답:
제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.
### Instruction:
answer to input

### Input:
CallCrazy 서비스는 어떤 것들을 배달하나요?

### Response:
We deliver food, Japanese food particularly, and recently we have expanded to logistics services.


**모델 저장**

In [ ]:
from huggingface_hub import login, HfApi
from google.colab import userdata

HF_TOKEN = userdata.get('Huggingface')
login(token=HF_TOKEN)

save_path = "fine-tuned-callcrazy-model"

# LoRA 적용된 모델 저장
trainer.model.save_pretrained(save_path)

# tokenizer 저장
trainer.processing_class.save_pretrained(save_path)

# Hugging Face Hub 업로드
repo_name = "JaeminKim/HF3B_callme"
api = HfApi()
api.create_repo(repo_name, exist_ok=True)

trainer.model.push_to_hub(repo_name)  # Model 업로드
trainer.processing_class.push_to_hub(repo_name)   # tokenizer 업로드

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

Saved model to https://huggingface.co/JaeminKim/HF3B_callme


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp1b384ogh/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

In [ ]:
!pip uninstall -y unsloth unsloth-zoo
!pip cache purge
!pip install -U --no-cache-dir "unsloth[colab-new]" unsloth-zoo

Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 175.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 239.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
from unsloth import FastLanguageModel

# Unsloth가 Hub repo에서 베이스 모델 + LoRA 어댑터를 자동으로 결합
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="JaeminKim/HF3B_callme",  # Fine-tuned repo
    max_seq_length=4096,
    dtype=None,          # None → bf16/fp16 자동 감지
    load_in_4bit=True,   # 학습 시와 동일 설정 유지
)

# 추론 모드 활성화 (Unsloth 내부 최적화 커널 적용)
FastLanguageModel.for_inference(model)

# device 확인만 (이동은 하지 않음)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"모델 디바이스: {device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Unsloth 2026.4.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


모델 디바이스: cuda


In [ ]:
def generate_response(prompt, max_length=512):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=max_length,
            eos_token_id=tokenizer.eos_token_id,
            temperature=0.7,  # 더 다양한 답변 생성
            top_p=0.9,  # 상위 90% 확률의 단어만 선택하여 일반적인 답변 방지
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# 테스트
instruction = "Answer to Input"
input_text = "What is CallCrazy service?"

alpaca_prompt = f"""제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""

response = generate_response(alpaca_prompt)
print(response)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
Answer to Input

### Input:
What is CallCrazy service?

### Response:
The CallCrazy service is a delivery service recently launched in Korea.


In [ ]:
instruction = "Answer to Input"
input_text = "What is the price of CallCrazy service?"

alpaca_prompt = f"""제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""

response = generate_response(alpaca_prompt)
print(response)

제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
Answer to Input

### Input:
What is the price of CallCrazy service?

### Response:
The basic fee is 5,000 won, and 1,000 won is added per km


In [ ]:
instruction = "Answer to Input"
input_text = "Explain about Seoul"

alpaca_prompt = f"""제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""

response = generate_response(alpaca_prompt)
print(response)

제공된 dataset은 CallCrazy라서 배달 서비스에 대한 Q&A 입니다. 주어진 데이터를 바탕으로 적절한 응답을 작성하세요.

### Instruction:
Answer to Input

### Input:
Explain about Seoul

### Response:
Seoul is the capital and largest city of Korea, known for its modern architecture, vibrant culture, and rich history.
